# 04 — Clean & integrate

Applies the decisions written down in `docs/QUALITY_FINDINGS.md` (Phase 3) — nothing new is
discovered here, only acted on. A row count is printed after every step ("provenance by print
statement"). Produces the three `data/processed/` tables used by `05_analysis.ipynb`.

In [1]:
import sys
import pandas as pd
import numpy as np

print("python", sys.version)
print("pandas", pd.__version__)

df_winter = pd.read_csv("../data/raw/311_2025-01_2025-02_downloaded_2026-07-12.csv", low_memory=False)
df_summer = pd.read_csv("../data/raw/311_2025-07_2025-08_downloaded_2026-07-12.csv", low_memory=False)
df = pd.concat([df_winter, df_summer], ignore_index=True)
print("311 raw combined rows:", len(df))

wx = pd.read_csv("../data/raw/weather_centralpark_2025_downloaded_2026-07-12.csv")
print("weather raw rows:", len(wx))

cat_map = pd.read_csv("../data/complaint_category_map.csv")
print("category mapping rows:", len(cat_map))

python 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
pandas 2.3.3


311 raw combined rows: 1223457
weather raw rows: 243
category mapping rows: 180


## 4.1 Clean 311

In [2]:
n_before = len(df)
df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
df["closed_date"] = pd.to_datetime(df["closed_date"], errors="coerce")
n_unparseable_created = df["created_date"].isna().sum()
print(f"unparseable created_date: {n_unparseable_created} (rows dropped for this reason: 0 -- none found)")
print(f"row count unchanged by date parsing: {len(df) == n_before}")

unparseable created_date: 0 (rows dropped for this reason: 0 -- none found)
row count unchanged by date parsing: True


In [3]:
n_before = len(df)
n_exact_dupes = df["unique_key"].duplicated().sum()
df = df.drop_duplicates(subset="unique_key", keep="first")
print(f"exact duplicate unique_key rows dropped: {n_before - len(df)} (Phase 3 found {n_exact_dupes})")
print(f"row count: {n_before} -> {len(df)}")

exact duplicate unique_key rows dropped: 0 (Phase 3 found 0)
row count: 1223457 -> 1223457


In [4]:
before_normalize = df["complaint_type"].copy()
df["complaint_type"] = df["complaint_type"].str.strip()
n_changed = (before_normalize != df["complaint_type"]).sum()
print(f"complaint_type values changed by whitespace-stripping: {n_changed} "
      "(Phase 3 already found 0 whitespace issues -- confirms nothing to fix)")
print("Case is NOT normalized: the controlled vocabulary in complaint_category_map.csv already "
      "lists both case variants (e.g. Elevator/ELEVATOR) as separate, correctly-mapped raw_type "
      "entries, so exact-match merging handles them without lowercasing the whole column.")

complaint_type values changed by whitespace-stripping: 0 (Phase 3 already found 0 whitespace issues -- confirms nothing to fix)
Case is NOT normalized: the controlled vocabulary in complaint_category_map.csv already lists both case variants (e.g. Elevator/ELEVATOR) as separate, correctly-mapped raw_type entries, so exact-match merging handles them without lowercasing the whole column.


In [5]:
n_before = len(df)
df = df.merge(cat_map, left_on="complaint_type", right_on="raw_type", how="left")
n_unmapped = df["category"].isna().sum()
print(f"rows after merging category mapping: {len(df)} (unchanged from {n_before}: {len(df) == n_before})")
print(f"unmapped rows: {n_unmapped}")
assert n_unmapped == 0, "found complaint_type values with no category mapping -- update complaint_category_map.csv"
df = df.drop(columns="raw_type")

rows after merging category mapping: 1223457 (unchanged from 1223457: True)
unmapped rows: 0


In [6]:
df["date"] = df["created_date"].dt.date

print("final 311_clean shape:", df.shape)
df.to_csv("../data/processed/311_clean.csv", index=False)
print("saved ../data/processed/311_clean.csv")

final 311_clean shape: (1223457, 14)


saved ../data/processed/311_clean.csv


## 4.2 Clean weather

In [7]:
wx["DATE"] = pd.to_datetime(wx["DATE"])
wx_clean = wx[["DATE", "TMAX", "TMIN", "PRCP", "SNOW"]].rename(columns={"DATE": "date"})

# units already verified as metric (Phase 2/3: TMAX range -7.1..37.2 C, plausible) -- no conversion needed.

full_calendar = pd.date_range(wx_clean["date"].min(), wx_clean["date"].max(), freq="D")
n_before = len(wx_clean)
wx_clean = wx_clean.set_index("date").reindex(full_calendar).rename_axis("date").reset_index()
n_missing_after_reindex = wx_clean["TMAX"].isna().sum()
print(f"weather rows: {n_before} -> {len(wx_clean)} after reindexing on the full calendar "
      f"({len(full_calendar)} days)")
print(f"missing days made visible as NaN: {n_missing_after_reindex}")

wx_clean["date"] = wx_clean["date"].dt.date
wx_clean.to_csv("../data/processed/weather_clean.csv", index=False)
print("saved ../data/processed/weather_clean.csv")

weather rows: 243 -> 243 after reindexing on the full calendar (243 days)
missing days made visible as NaN: 0
saved ../data/processed/weather_clean.csv


## 4.3 Aggregate + join

In [8]:
daily_counts = df.groupby(["date", "category"]).size().reset_index(name="n_requests")
print("aggregated (date, category) rows:", len(daily_counts))
print("distinct dates in 311 subset:", daily_counts["date"].nunique())
print("distinct categories:", daily_counts["category"].nunique())

aggregated (date, category) rows: 1143
distinct dates in 311 subset: 121
distinct categories: 10


In [9]:
n_before_join = len(daily_counts)
analysis_daily = daily_counts.merge(wx_clean, on="date", how="left")
print(f"row count before join: {n_before_join}, after left join: {len(analysis_daily)} "
      f"(unchanged: {len(analysis_daily) == n_before_join})")

analysis_daily["date"] = pd.to_datetime(analysis_daily["date"])
analysis_daily["weekday"] = analysis_daily["date"].dt.day_name()
analysis_daily["is_weekend"] = analysis_daily["date"].dt.dayofweek >= 5

analysis_daily.to_csv("../data/processed/analysis_daily.csv", index=False)
print("saved ../data/processed/analysis_daily.csv")
analysis_daily.head()

row count before join: 1143, after left join: 1143 (unchanged: True)
saved ../data/processed/analysis_daily.csv


,date,category,n_requests,TMAX,TMIN,PRCP,SNOW,weekday,is_weekend
0,2025-01-01,BUILDING,873,10.6,3.9,0.0,0.0,Wednesday,False
1,2025-01-01,FLOODING,40,10.6,3.9,0.0,0.0,Wednesday,False
2,2025-01-01,HEAT,951,10.6,3.9,0.0,0.0,Wednesday,False
3,2025-01-01,NOISE,5539,10.6,3.9,0.0,0.0,Wednesday,False
4,2025-01-01,OTHER,483,10.6,3.9,0.0,0.0,Wednesday,False


## 4.4 Validate the join

In [10]:
# 1. row count unchanged by the join
print("Check 1 -- row count unchanged by join:",
      len(analysis_daily) == len(daily_counts), f"({len(analysis_daily)} == {len(daily_counts)})")

# 2. n_requests sum == cleaned event count
sum_n_requests = analysis_daily["n_requests"].sum()
n_cleaned_events = len(df)
print("Check 2 -- n_requests sum == cleaned 311 event count:",
      sum_n_requests == n_cleaned_events, f"({sum_n_requests} == {n_cleaned_events})")

# 3. % days with complete weather
days_total = analysis_daily["date"].nunique()
days_with_weather = analysis_daily.dropna(subset=["TMAX"])["date"].nunique()
print(f"Check 3 -- days with complete weather: {days_with_weather}/{days_total} "
      f"({days_with_weather/days_total*100:.1f}%)")

# 4. spot check: the biggest snow day in-window, 2025-02-08 (SNOW = 76.0 mm)
spot = analysis_daily[analysis_daily["date"] == "2025-02-08"]
print("\nCheck 4 -- spot check 2025-02-08 (known biggest in-window snow day):")
print(spot[["date", "category", "n_requests", "TMAX", "TMIN", "PRCP", "SNOW", "weekday", "is_weekend"]])
print(f"SNOW value matches raw weather file (76.0 mm): {(spot['SNOW'] == 76.0).all()}")

Check 1 -- row count unchanged by join: True (1143 == 1143)
Check 2 -- n_requests sum == cleaned 311 event count: True (1223457 == 1223457)
Check 3 -- days with complete weather: 121/121 (100.0%)

Check 4 -- spot check 2025-02-08 (known biggest in-window snow day):
          date    category  n_requests  TMAX  TMIN  PRCP  SNOW   weekday  \
379 2025-02-08    BUILDING        1039   1.7  -3.2  15.5  76.0  Saturday   
380 2025-02-08    FLOODING          48   1.7  -3.2  15.5  76.0  Saturday   
381 2025-02-08        HEAT        1555   1.7  -3.2  15.5  76.0  Saturday   
382 2025-02-08       NOISE        1690   1.7  -3.2  15.5  76.0  Saturday   
383 2025-02-08       OTHER         384   1.7  -3.2  15.5  76.0  Saturday   
384 2025-02-08     PARKING        2045   1.7  -3.2  15.5  76.0  Saturday   
385 2025-02-08  SANITATION         445   1.7  -3.2  15.5  76.0  Saturday   
386 2025-02-08    SNOW_ICE          47   1.7  -3.2  15.5  76.0  Saturday   
387 2025-02-08      SOCIAL         239   1.7  -3.2

## 4.5 Next step

`data/processed/analysis_daily.csv` is the table `05_analysis.ipynb` works from. Its structure is
documented in `docs/data_dictionary_analysis_daily.md`.